# Traffic Sensor Data Cleaning Pipeline

Below is the test cell to read all the Excel files

In [47]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "DDWEB_Downloads"
excel_files = sorted(DATA_DIR.glob("*.xlsx"))

print(f"Found {len(excel_files)} Excel files")
for f in excel_files:
    print(f.name)

Found 8 Excel files
DDweb_Auftrag_31032026_1706.xlsx
DDweb_Standort_01042026_1658.xlsx
DDweb_VI_Rohdaten_01042026_1726.xlsx
DDweb_VI_Rohdaten_01042026_1727.xlsx
DDweb_VI_Rohdaten_16102023_1722.xlsx
DDweb_VI_Rohdaten_31032026_1633.xlsx
DDweb_VI_Rohdaten_31032026_1642.xlsx
DDweb_VI_Rohdaten_31032026_1656.xlsx


In [48]:
df_Auftrag = pd.read_excel(excel_files[0])
df_Auftrag.head()

/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Startdatum,Enddatum,Beschreibung,Geräte-ID,Gerätetyp,Standorttitel,Stadt,Inhaber,Erstellt
0,2022-01-28 13:00:00,2023-04-29 11:00:00,Fahrtrichtung Nord,6429,DD.plus,Albanstraße Nr. 23 DD 6429,Berlin Tempelhof-Schöneberg,NaN,2022-02-28 13:23:57.410
1,2022-02-25 14:00:00,2023-03-18 08:00:00,Fahrtrichtung-Süd,5949,DD.plus,Bahnstraße Nr. 10 DD 5949,Berlin Tempelhof-Schöneberg,NaN,2022-02-28 13:26:39.890
2,2021-04-27 11:10:00,2049-01-01 00:00:00,i.H.-HNr. 4,5951,DD.plus,Boelkestraße Nr. 58 Fr-Ri. Süd DD 5951,Berlin -Tempelhof-Schöneberg,NaN,2023-07-28 15:41:38.778
3,2021-05-04 11:10:00,2049-01-01 00:00:00,Fahrtrichtung Nord,5950,DD.plus,Boelkestraße Nr. 65 Fr. Nord DD 5950,Berlin Tempelhof-Schöneberg,NaN,2021-05-05 10:12:33.113
4,2026-03-13 13:00:00,2049-01-01 00:00:00,DD 5949 Halker Zeile,5949,DD.plus,DD 5949 Halker Zeile,Berlin Tempelhof-Schöneberg,NaN,2026-03-17 09:44:25.166


In [49]:
# ── QA accumulator ──────────────────────────────────────────────────────────
# We populate this dict throughout the notebook; it is printed as a report at the end.
qa_report = {}

print("Configuration loaded.")

Configuration loaded.


In [50]:
# Ensure date columns are proper datetimes, not strings
df_Auftrag["Startdatum"] = pd.to_datetime(df_Auftrag["Startdatum"], errors="coerce")
df_Auftrag["Enddatum"]   = pd.to_datetime(df_Auftrag["Enddatum"],   errors="coerce")

#the Startdatum column — the date each deployment began. That range makes perfect sense: the oldest sensor was first deployed in 2017,
#and the most recently started deployment kicked off in March 2026.
#Sentinel dates used in Auftrag to mean "deployment still active"
ACTIVE_SENTINELS = {
    pd.Timestamp("2049-01-01"),
    pd.Timestamp("2100-01-01"),
}

df_Auftrag["is_active"] = df_Auftrag["Enddatum"].apply(
    lambda d: any(abs((d - s).days) < 2 for s in ACTIVE_SENTINELS) if pd.notna(d) else False
)
df_Auftrag["Enddatum_clean"] = df_Auftrag.apply(
    lambda row: pd.NaT if row["is_active"] else row["Enddatum"], axis=1
)

print(f"Auftrag rows: {len(df_Auftrag)}")
print(f"Unique device IDs in Auftrag: {df_Auftrag['Geräte-ID'].nunique()}")

Auftrag rows: 44
Unique device IDs in Auftrag: 30


In [51]:
df_Standort = pd.read_excel(excel_files[1])
df_Standort.head()
print(f"Standort rows: {len(df_Standort)}")

Standort rows: 45


---
## 1. Drop Unimplemented Columns on Ingestion

Five columns are structurally zero across all files because the corresponding
sensor features were never activated. They are dropped immediately on load
to keep the working dataframe clean. The reasons are documented here:

| Column | Reason for dropping |
|---|---|
| `Schall (dB)` | Always 0 — sound measurement not implemented |
| `Abstand (cm)` | Always 0 — lateral distance not implemented |
| `Fahrspur` | Always 0 — lane indicator, not applicable on single-lane residential streets |
| `Geschwindigkeit (km/h)` | Always 0 — point speed not implemented; actual speed is in entry/exit columns |
| `Richtung` | Always 1 — one device per direction so this carries no information |

In [52]:
raw_files = sorted(DATA_DIR.glob("DDweb_VI_Rohdaten_*.xlsx"))
print(f"Found {len(raw_files)} raw data file(s):")
for f in raw_files:
    print(f"  {f.name}")

Found 6 raw data file(s):
  DDweb_VI_Rohdaten_01042026_1726.xlsx
  DDweb_VI_Rohdaten_01042026_1727.xlsx
  DDweb_VI_Rohdaten_16102023_1722.xlsx
  DDweb_VI_Rohdaten_31032026_1633.xlsx
  DDweb_VI_Rohdaten_31032026_1642.xlsx
  DDweb_VI_Rohdaten_31032026_1656.xlsx


In [53]:
COLS_TO_DROP = [
    "Schall (dB)",
    "Abstand (cm)",
    "Fahrspur",
    "Geschwindigkeit (km/h)",
    "Richtung",
]

RENAME_MAP = {
    "Geräte-ID":                          "device_id",
    "Datum":                              "datum_raw",
    "Eintrittsgeschwindigkeit (km/h)":    "speed_entry",
    "Austrittsgeschwindigkeit (km/h)":    "speed_exit",
    "Länge (dm)":                         "laenge_dm",
    "Klasse":                             "klasse",
    "Fahrzeugklassen-Bezeichnung":        "klasse_label",
}

frames = []
for fpath in raw_files:
    _df = pd.read_excel(fpath, dtype={"Geräte-ID": str})
    _df["source_file"] = fpath.name          # keep provenance
    _df = _df.drop(columns=COLS_TO_DROP, errors="ignore")
    _df = _df.rename(columns=RENAME_MAP)
    frames.append(_df)

df = pd.concat(frames, ignore_index=True)

print(f"Combined dataframe: {len(df):,} rows × {df.shape[1]} columns")
print(f"Columns kept: {list(df.columns)}")
df.head(3)

/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Combined dataframe: 134,070 rows × 8 columns
Columns kept: ['device_id', 'datum_raw', 'speed_entry', 'speed_exit', 'laenge_dm', 'klasse', 'klasse_label', 'source_file']


/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/djiang/Berlin_traffic_data/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,device_id,datum_raw,speed_entry,speed_exit,laenge_dm,klasse,klasse_label,source_file
0,5951,2026-03-25 00:19:54,62,70,42,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx
1,5951,2026-03-25 00:21:52,49,51,38,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx
2,5951,2026-03-25 00:51:22,37,43,38,7,Pkw,DDweb_VI_Rohdaten_01042026_1726.xlsx


---
## 2. Timestamp Parsing with Multi-Format Detection

Two timestamp formats exist in the wild across export batches:
- **ISO format** (`YYYY-MM-DD HH:MM:SS`) — used in all recent exports; pandas reads this automatically
- **Legacy string format** (`DD/MM/YYYY HH:MM:SS`) — used in the 2022/2023 export; pandas will
  silently misparse this if you call `to_datetime` without specifying `dayfirst=True`

plan: detect the format per file and parse accordingly, then extract `datum` (date) and
`stunde` (hour 0–23) as separate columns for aggregation.

In [54]:
def parse_datum_column(series: pd.Series) -> pd.Series:
    """
    Parse a Datum column that may contain either:
    - datetime64 objects (already parsed by pandas on load from recent files)
    - strings in 'DD/MM/YYYY HH:MM:SS' format (legacy export format)
    Returns a datetime64 Series. Values that cannot be parsed become NaT.
    """
    if pd.api.types.is_datetime64_any_dtype(series):
        return series

    parsed = pd.to_datetime(series, dayfirst=True, errors="coerce") # It's a string column. Try the legacy DD/MM/YYYY format first (dayfirst=True)
    return parsed


# Parse each source file's dates separately
parsed_parts = []
for fpath in raw_files:
    mask = df["source_file"] == fpath.name
    parsed_parts.append(parse_datum_column(df.loc[mask, "datum_raw"]))

df["datum_parsed"] = pd.concat(parsed_parts).sort_index()

# For time components analysis
df["datum"]  = df["datum_parsed"].dt.date           # calendar date - for daily aggregation
df["stunde"] = df["datum_parsed"].dt.hour           # for hourly aggregation
df["wochentag"] = df["datum_parsed"].dt.day_name()  # for weekday patterns

# Flag any rows where parsing failed
df["flag_unparseable_timestamp"] = df["datum_parsed"].isna()
n_bad_ts = df["flag_unparseable_timestamp"].sum()
qa_report["unparseable_timestamp"] = int(n_bad_ts)


print(f"Timestamp parsing complete.")
print(f"  Rows with unparseable timestamp (flagged, not dropped): {n_bad_ts}")
print(f"  Date range: {df['datum_parsed'].min()} → {df['datum_parsed'].max()}")
df[["source_file", "datum_raw", "datum_parsed", "datum", "stunde", "wochentag"]].head(4)

Timestamp parsing complete.
  Rows with unparseable timestamp (flagged, not dropped): 0
  Date range: 2022-09-02 15:26:34 → 2026-03-31 23:57:06


,source_file,datum_raw,datum_parsed,datum,stunde,wochentag
0,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:19:54,2026-03-25 00:19:54,2026-03-25,0,Wednesday
1,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:21:52,2026-03-25 00:21:52,2026-03-25,0,Wednesday
2,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 00:51:22,2026-03-25 00:51:22,2026-03-25,0,Wednesday
3,DDweb_VI_Rohdaten_01042026_1726.xlsx,2026-03-25 01:25:16,2026-03-25 01:25:16,2026-03-25,1,Wednesday


---
## 3. Validate Geräte-ID Against Reference Table

Every `Geräte-ID` in the raw data should appear in the Auftrag deployment table.
A mismatch means either (a) the sensor was deployed without being registered, or
(b) the georeferencing table has not been updated after a device was moved or replaced.
Flagged rows cannot be georeferenced and should be held back from spatial analysis.

In [55]:
# Build the set of all device IDs that have ever appeared in the Auftrag table.
known_device_ids = set(df_Auftrag["Geräte-ID"].astype(str).unique())

df["unknown_device_flagged"] = ~df["device_id"].astype(str).isin(known_device_ids)

unknown_ids = df.loc[df["unknown_device_flagged"], "device_id"].unique()
n_unknown_rows = df["unknown_device_flagged"].sum()
qa_report["unknown_device_id"] = int(n_unknown_rows)

print(f"Known device IDs in Auftrag: {len(known_device_ids)}")
print(f"Unique device IDs in raw data: {df['device_id'].nunique()}")
print(f"Unregistered device IDs: {list(unknown_ids)}")
print(f"Rows flagged as unknown device (not dropped): {n_unknown_rows:,}")

Known device IDs in Auftrag: 30
Unique device IDs in raw data: 4
Unregistered device IDs: []
Rows flagged as unknown device (not dropped): 0


In [56]:
# Helpful summary: which devices appear in raw data, and whether they're known
device_summary = (
    df.groupby("device_id")
    .agg(row_count=("datum_parsed", "count"))
    .assign(in_auftrag=lambda x: x.index.isin(known_device_ids))
    .sort_values("row_count", ascending=False)
)
display(device_summary)

,row_count,in_auftrag
device_id,,
6426,54424,True
5950,41020,True
6429,36020,True
5951,2606,True


---
## 4. Speed Plausibility Check

Speed is recorded as entry speed and exit speed. Flagged rows are logged but won't be dropped for this step. 


In [57]:
# ── Speed plausibility thresholds (adjustable) ───────────────────────────────────────────
SPEED_MAX_MOTORISED  = 150   # km/h — flag any motorised vehicle above this
SPEED_MIN_MOTORISED  = 1     # km/h — flag any motorised vehicle below this (fyi. under 10km/h might be unreliable)
SPEED_MAX_BICYCLE    = 50    # km/h — flag any bicycle above this

# Klasse codes that are motorised (i.e. NOT bicycle or unknown).
MOTORISED_CLASSES = {2, 3, 5, 7, 8, 9, 10, 11, 64}
BICYCLE_CLASS     = 230

def speed_flag(speed_col, klasse_col):
    """
    It will return True for rows where the speed value is implausible given the vehicle class.
    Motorised vehicles: flag if speed < SPEED_MIN_MOTORISED or > SPEED_MAX_MOTORISED
    Bicycles:          flag if speed > SPEED_MAX_BICYCLE
    At the same time, zero exit speed is flagged — it represents an undefined read, not a stopped vehicle.
    """
    is_motorised = klasse_col.isin(MOTORISED_CLASSES)
    is_bicycle   = klasse_col == BICYCLE_CLASS

    flagged_motor   = is_motorised & (
        (speed_col > SPEED_MAX_MOTORISED) | (speed_col < SPEED_MIN_MOTORISED)
    )
    flagged_bicycle = is_bicycle & (speed_col > SPEED_MAX_BICYCLE)
    flag_zero_exit    = speed_col == 0

    return flagged_motor | flagged_bicycle | flag_zero_exit

In [58]:
df["speed_entry_flagged"] = speed_flag(df["speed_entry"], df["klasse"])
df["speed_exit_flagged"] = speed_flag(df["speed_exit"], df["klasse"])

# A row is flagged if either entry or exit speed is flagged
df["flag_speed"] = df["speed_entry_flagged"] | df["speed_exit_flagged"]

n_speed_entry = df["speed_entry_flagged"].sum()
n_speed_exit  = df["speed_exit_flagged"].sum()
n_speed_any   = df["flag_speed"].sum()
qa_report["implausible_entry_speed"] = int(n_speed_entry)
qa_report["implausible_exit_speed"]  = int(n_speed_exit)
qa_report["implausible_speed_any"]   = int(n_speed_any)

print(f"Rows with implausible entry speed: {n_speed_entry:,}")
print(f"Rows with implausible exit speed:  {n_speed_exit:,}")
print(f"Rows with implausible speed (either): {n_speed_any:,}")

Rows with implausible entry speed: 0
Rows with implausible exit speed:  1
Rows with implausible speed (either): 1


In [59]:
if n_speed_any > 0:
    print("\nSample flagged rows (investigate before the exclusion):")
    display(df[df["flag_speed"]][
        ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "source_file"]
        ].head(10)
        )


Sample flagged rows (investigate before the exclusion):


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,source_file
27742,5950,2026-03-29 09:02:25,11,Lfw,25,0,DDweb_VI_Rohdaten_01042026_1727.xlsx


---
## 5. Duplicate Detection

A true duplicate is a row where essential/meaningful field is identical — same device,
same timestamp, same class, same speed, same length -> This might indicate a data transmission error 
(the device sent the same record twice).
Flagged but not dropped — review before removing.

In [60]:
DEDUP_COLS = [
    "device_id",
    "datum_parsed",   # timestamp to the second
    "klasse",
    "speed_entry",
    "speed_exit",
    "laenge_dm",
]

# keep=False marks ALL copies of a duplicate as True, so the full set of duplicated rows is visible for investigation.
df["flag_duplicate"] = df.duplicated(subset=DEDUP_COLS, keep=False)
n_dup = df["flag_duplicate"].sum()
qa_report["duplicate_rows"] = int(n_dup)

print(f"Rows flagged as duplicates: {n_dup:,}")

if n_dup > 0:
    print("\nDuplicate groups (sorted by device and timestamp):")
    display(
        df[df["flag_duplicate"]]
        .sort_values(["device_id", "datum_parsed"])
        [["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "laenge_dm", "source_file"]]
        .head(60)
    )

Rows flagged as duplicates: 120

Duplicate groups (sorted by device and timestamp):


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,laenge_dm,source_file
43799,6426,2022-09-02 16:08:36,230,Fahrrad,13,12,16,DDweb_VI_Rohdaten_16102023_1722.xlsx
43800,6426,2022-09-02 16:08:36,230,Fahrrad,13,12,16,DDweb_VI_Rohdaten_16102023_1722.xlsx
44234,6426,2022-09-02 18:00:04,230,Fahrrad,20,21,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
44235,6426,2022-09-02 18:00:04,230,Fahrrad,20,21,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
44591,6426,2022-09-02 19:50:56,230,Fahrrad,15,14,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
44592,6426,2022-09-02 19:50:56,230,Fahrrad,15,14,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
45997,6426,2022-09-03 16:31:46,230,Fahrrad,14,13,14,DDweb_VI_Rohdaten_16102023_1722.xlsx
45998,6426,2022-09-03 16:31:46,230,Fahrrad,14,13,14,DDweb_VI_Rohdaten_16102023_1722.xlsx
46792,6426,2022-09-07 16:21:40,230,Fahrrad,15,13,15,DDweb_VI_Rohdaten_16102023_1722.xlsx
46793,6426,2022-09-07 16:21:40,230,Fahrrad,15,13,15,DDweb_VI_Rohdaten_16102023_1722.xlsx


---
## 6. Entry/Exit Speed Delta Check

A large difference between entry and exit speed (even though both speeds are plausible) can indicate a sensor read error. A legitimate vehicle
passage should show similar entry and exit speeds on a residential street, meaning the speed should not change dramatically.

In [61]:
SPEED_DELTA_MAX      = 40    # km/h — flag entry/exit speed difference above this

df["speed_delta"] = (df["speed_entry"] - df["speed_exit"]).abs()

df["flag_speed_delta"] = df["speed_delta"] > SPEED_DELTA_MAX

n_delta = df["flag_speed_delta"].sum()
qa_report["large_speed_delta"] = int(n_delta)

print(f"Rows where |entry − exit| > {SPEED_DELTA_MAX} km/h: {n_delta:,}")

if n_delta > 0:
    print("\nBreakdown by vehicle class:")
    delta_by_class = (
        df[df["flag_speed_delta"]]
        .groupby(["klasse", "klasse_label"])
        .agg(
            flagged_rows=("speed_delta", "count"),
            max_delta=("speed_delta", "max"),
            mean_delta=("speed_delta", "mean"),
        )
        .round(1)
        .sort_values("flagged_rows", ascending=False)
    )
    display(delta_by_class)

Rows where |entry − exit| > 40 km/h: 9

Breakdown by vehicle class:


,,flagged_rows,max_delta,mean_delta
klasse,klasse_label,,,
7,Pkw,4,101,58.25
10,Krad,3,57,51.333333
3,Lkw,1,43,43.0
11,Lfw,1,48,48.0


In [62]:
print("\nSample flagged rows:")
display(
    df[df["flag_speed_delta"]][
    ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "speed_delta"]
    ].sort_values("speed_delta", ascending=False).head(15)
)


Sample flagged rows:


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,speed_delta
43626,6426,2022-09-02 15:26:34,7,Pkw,111,10,101
1672,5951,2026-03-29 16:01:54,10,Krad,28,85,57
1489,5951,2026-03-29 00:33:12,10,Krad,29,83,54
62479,6426,2022-09-14 15:45:50,11,Lfw,18,66,48
8706,5950,2026-03-25 22:22:48,7,Pkw,13,59,46
27378,5950,2026-03-29 00:13:30,7,Pkw,26,71,45
29426,5950,2026-03-29 16:17:14,10,Krad,20,63,43
61871,6426,2022-09-14 09:36:34,3,Lkw,12,55,43
31061,5950,2026-03-30 01:25:08,7,Pkw,37,78,41


---
## 7. Unclassifiable Vehicles (Klasse 6 and Klasse 250)

Two classification codes indicate problematic records:

- **Klasse 6** (`nk Kfz` — nicht klassifizierbar): the sensor could not classify the vehicle.
  A high rate per sensor suggests an alignment or obstruction problem.
- **Klasse 250 (partially obscured)** (`Kfz`)

In [63]:
UNCLASSIFIABLE_CODES = {6, 250}   # row-level unclassifiable

df["flag_unclassifiable"]   = df["klasse"].isin(UNCLASSIFIABLE_CODES)
n_unclass  = df["flag_unclassifiable"].sum()
qa_report["unclassifiable_rows"]    = int(n_unclass)

print(f"Klasse 6 / 250 (unclassifiable, row-level): {n_unclass:,} rows")

Klasse 6 / 250 (unclassifiable, row-level): 11 rows


In [64]:
# Per-sensor breakdown for Klasse 6/250 — high rates indicate hardware problems
if n_unclass > 0:
    print("\nKlasse 6/250 per device (as % of that device's total rows):")
    device_total = df.groupby("device_id").size().rename("total")
    device_unclass = df[df["flag_unclassifiable"]].groupby("device_id").size().rename("unclassifiable")
    unclass_rate = pd.concat([device_total, device_unclass], axis=1).fillna(0)
    unclass_rate["pct"] = (unclass_rate["unclassifiable"] / unclass_rate["total"] * 100).round(2)
    display(unclass_rate[unclass_rate["unclassifiable"] > 0].sort_values("pct", ascending=False))



Klasse 6/250 per device (as % of that device's total rows):


,total,unclassifiable,pct
device_id,,,
6426,54424,9.0,0.02
5950,41020,2.0,0.00


---
## 8. Multi-Location Deployment Check

A single device can only be in one place at a time - meaning the Auftrag table
records two overlapping deployment windows for the same `Geräte-ID` at different
locations.

This check operates entirely on the **Auftrag reference table**, not on the raw
sensor readings. It is run after reference tables are loaded, so no raw data is
needed. A clean result here means every reading can be unambiguously assigned to
exactly one location for any given timestamp.

In [66]:
# Check for overlapping deployment windows per device ──────────────
# Strategy: for each device, compare every pair of its deployment windows.
# Using the cleaned Enddatum (where sentinels are replaced with pd.Timestamp.max) to have open-ended active deployments are included in the comparison correctly.
overlap_records = []

for gid, group in df_Auftrag.groupby("Geräte-ID"):
    rows = group.sort_values("Startdatum").reset_index(drop=True)
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            a_start = rows.loc[i, "Startdatum"]
            a_end   = rows.loc[i, "Enddatum_clean"]
            b_start = rows.loc[j, "Startdatum"]
            b_end   = rows.loc[j, "Enddatum_clean"]

            if pd.isna(a_start) or pd.isna(b_start):
                continue # Skip if either start date is NaT

            if a_start < b_end and b_start < a_end: # Overlap condition: intervals [a_start, a_end) and [b_start, b_end) intersect
                overlap_records.append({
                    "device_id":      gid,
                    "location_A":     rows.loc[i, "Standorttitel"],
                    "start_A":        a_start,
                    "end_A":          rows.loc[i, "Enddatum"],   # show the raw end date
                    "location_B":     rows.loc[j, "Standorttitel"],
                    "start_B":        b_start,
                    "end_B":          rows.loc[j, "Enddatum"],
                    "overlap_start":  max(a_start, b_start),
                    "overlap_end":    min(a_end,   b_end),
                })

n_overlaps = len(overlap_records)
qa_report["overlapping_deployment_windows"] = n_overlaps

if n_overlaps == 0:
    print("✓ No overlapping deployment windows found. Every device has clean, non-overlapping location history.")
else:
    print(f"{n_overlaps} overlapping deployment window pair detected:\n")
    overlap_df = pd.DataFrame(overlap_records)
    display(overlap_df)
    print("\nThese pairs indicate a data-entry error in the Auftrag table.")
    print("Readings that fall in the overlap period cannot be unambiguously")
    print("assigned to a single location. Investigate before spatial analysis.")

✓ No overlapping deployment windows found. Every device has clean, non-overlapping location history.


In [67]:
# A further step: flagging any raw sensor readings that fall inside an overlap period

def in_any_overlap(row, overlaps):
    gid = str(row["device_id"])
    ts  = row["datum_parsed"]
    if pd.isna(ts):
        return False
    return any(
        str(o["device_id"]) == gid and o["overlap_start"] <= ts <= o["overlap_end"]
        for o in overlaps
    )

df["flag_ambiguous_location"] = df.apply(in_any_overlap, overlaps=overlap_records, axis=1)
n_amb = df["flag_ambiguous_location"].sum()
qa_report["step9_raw_rows_in_overlap_window"] = int(n_amb)
print(f"\nRaw sensor rows falling inside an overlap period: {n_amb:,}")

# If no overlaps, add the column as all-False for consistency with the flag consolidation step
if "flag_ambiguous_location" not in df.columns:
    df["flag_ambiguous_location"] = False
    qa_report["raw_rows_in_overlap_window"] = 0


Raw sensor rows falling inside an overlap period: 0


---
## Consolidate Flag Columns & Build Flag Detail Table

We add a single `any_flag` boolean column so filtered analyses can easily
exclude all suspect rows in one expression. We also build a tidy `flag_detail`
table containing only the flagged rows, with a `flag_reasons` column listing
which checks triggered.

In [ ]:
# All flag columns written during this notebook
FLAG_COLS = [
    "flag_unparseable_timestamp",
    "flag_unknown_device",
    "flag_unclassifiable",
    "flag_speed_entry",
    "flag_speed_exit",
    "flag_speed_any",
    "flag_duplicate",
    "flag_speed_delta",
]

# Keep only the flag columns that actually exist (defensive in case of future changes)
active_flags = [c for c in FLAG_COLS if c in df.columns]

df["any_flag"] = df[active_flags].any(axis=1)

# Build a human-readable 'flag_reasons' string per row
def summarise_flags(row):
    triggered = [c.replace("flag_", "") for c in active_flags if row.get(c)]
    return "; ".join(triggered) if triggered else ""

df["flag_reasons"] = df.apply(summarise_flags, axis=1)

# flag_detail: all rows with at least one flag, with key columns for investigation
flag_detail = df[df["any_flag"]][
    ["device_id", "datum_parsed", "klasse", "klasse_label", "speed_entry", "speed_exit", "speed_delta", "laenge_dm", "source_file", "flag_reasons"]
    ].copy()

print(f"Total flagged rows (any flag): {df['any_flag'].sum():,} of {len(df):,} "
      f"({df['any_flag'].mean()*100:.1f}%)")
print(f"\nFlag detail table: {len(flag_detail):,} rows")
flag_detail.head(50)

Total flagged rows (any flag): 140 of 134,070 (0.1%)

Flag detail table: 140 rows


,device_id,datum_parsed,klasse,klasse_label,speed_entry,speed_exit,speed_delta,laenge_dm,source_file,flag_reasons
1489,5951,2026-03-29 00:33:12,10,Krad,29,83,54,24,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
1672,5951,2026-03-29 16:01:54,10,Krad,28,85,57,17,DDweb_VI_Rohdaten_01042026_1726.xlsx,speed_delta
8706,5950,2026-03-25 22:22:48,7,Pkw,13,59,46,38,DDweb_VI_Rohdaten_01042026_1727.xlsx,speed_delta
16814,5950,2026-03-27 09:34:00,6,nk Kfz,23,15,8,35,DDweb_VI_Rohdaten_01042026_1727.xlsx,unclassifiable
27378,5950,2026-03-29 00:13:30,7,Pkw,26,71,45,42,DDweb_VI_Rohdaten_01042026_1727.xlsx,speed_delta
29426,5950,2026-03-29 16:17:14,10,Krad,20,63,43,23,DDweb_VI_Rohdaten_01042026_1727.xlsx,speed_delta
31061,5950,2026-03-30 01:25:08,7,Pkw,37,78,41,44,DDweb_VI_Rohdaten_01042026_1727.xlsx,speed_delta
35387,5950,2026-03-30 16:16:16,6,nk Kfz,13,10,3,49,DDweb_VI_Rohdaten_01042026_1727.xlsx,unclassifiable
43626,6426,2022-09-02 15:26:34,7,Pkw,111,10,101,42,DDweb_VI_Rohdaten_16102023_1722.xlsx,speed_delta
43799,6426,2022-09-02 16:08:36,230,Fahrrad,13,12,1,16,DDweb_VI_Rohdaten_16102023_1722.xlsx,duplicate


---
## QA Report Summary

A single-glance summary of all checks run during this pipeline.

In [ ]:
qa_report["total_rows_ingested"]  = len(df)
qa_report["total_rows_clean"]     = int((~df["any_flag"]).sum())
qa_report["source_files_loaded"]  = len(raw_files)

# Print as a tidy table
qa_df = pd.DataFrame.from_dict(qa_report, orient="index", columns=["count"])

STEP_LABELS = {
    "source_files_loaded":                    "Files loaded",
    "total_rows_ingested":                    "Total rows ingested",
    "total_rows_clean":                       "Rows with no flags",
    "total_rows_any_flag":                    "Rows with ≥1 flag",
    "unparseable_timestamp":                  "Unparseable timestamps",
    "unknown_device_id":                      "Rows with unregistered Geräte-ID",
    "unclassifiable_rows":                    "Rows Klasse 6/250 (unclassifiable)",
    "implausible_entry_speed":                "Implausible entry speed",
    "implausible_exit_speed":                 "Implausible exit speed",
    "implausible_speed_any":                  "Implausible speed (entry or exit)",
    "duplicate_rows":                         "Duplicate rows (all copies flagged)",
    "large_speed_delta":                      "Large entry/exit speed delta",
}

qa_df.index = [STEP_LABELS.get(k, k) for k in qa_df.index]
display(qa_df.style.format("{:,}"))

,count
Unparseable timestamps,0
Rows with unregistered Geräte-ID,0
Implausible entry speed,0
Implausible exit speed,1
Implausible speed (entry or exit),1
Duplicate rows (all copies flagged),120
step10_large_speed_delta,9
Rows Klasse 6/250 (unclassifiable),11
Total rows ingested,"134,070"
Rows with no flags,"133,930"
